# Healthcare LoRA SFT — training

Runs on a **free Colab T4**. Set `Runtime > Change runtime type > T4 GPU` before running anything.

Pipeline: clone repo -> install -> rebuild data -> smoke test -> full train -> evaluate -> download adapter.

In [ ]:
!nvidia-smi

## 1. Clone and install

**Edit the URL below to your repository before running.**

`torch` is deliberately absent from `requirements-train.txt` — Colab already ships a CUDA build, and reinstalling it usually breaks the environment.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/healthcare-sft.git
%cd healthcare-sft
!pip install -q -r requirements-train.txt

## 2. Rebuild the dataset

The repo already contains `data/`, but regenerating proves the pipeline is reproducible from source rather than from a committed artifact.

Expect: 22 labels, 2040 training examples, 11 dropped as too long, 3 dropped as train/test duplicates.

In [ ]:
!python -m src.data_prep

## 3. Smoke test — 10 examples, 1 epoch

Catches OOM, dtype and masking errors in ~60 seconds instead of 30 minutes in.

**Check `trainable params` reports roughly 1%.** If it says 100%, LoRA did not attach and you are about to full-fine-tune by accident.

In [ ]:
!python -m src.train --limit 10 --epochs 1 --output-dir /tmp/smoke

## 4. Full training run

765 optimizer steps (2040 examples / (2 x 4) per step x 3 epochs), roughly 20-40 minutes.

Watch that the loss falls and never becomes `nan` — `nan` is the classic fp16 instability symptom.

If it OOMs: `--batch-size 1`, then `--max-seq-length 768`.

In [ ]:
!python -m src.train

## 5. Evaluate — base vs tuned

Two runs over one code path. Identical prompts from `src/prompts.py`, identical greedy decoding, identical 4-bit quantization, identical held-out test sets. The **only** difference is whether the adapter is attached.

Run the base cell first — it is the number the tuned model has to beat.

In [ ]:
!python -m src.evaluate --no-adapter --tag base

In [ ]:
!python -m src.evaluate --adapter adapters/qwen-healthcare-lora --tag tuned

## 6. Download the adapter and results

The adapter is tens of MB: LoRA matrices only, not the 3 GB base model. Unzip `adapters/` and `results/` into the repo and commit them.

In [ ]:
!du -sh adapters/qwen-healthcare-lora
!zip -r artifacts.zip adapters/ results/
from google.colab import files
files.download('artifacts.zip')